<a href="https://colab.research.google.com/github/Edison-great/character-card-styles/blob/main/Ps1cho's_Invoke_AI_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Ps1cho's InvokeAI Notebook**

just execute everything in order and you should be fine

## 📱 Cell 0 — Anti-Sleep (mobile only)
> **Run this first if you're on a phone/tablet.** It prevents the browser from killing the Colab tab in the background.
>
> Uses a silent audio loop + periodic JS pings to keep the tab alive. Safe to skip on desktop.

In [ ]:
from IPython.display import display, HTML

display(HTML('''
<div id="keep-alive-status" style="
    padding: 8px 14px;
    background: #1a1a2e;
    color: #0f0;
    border-radius: 6px;
    font-family: monospace;
    font-size: 13px;
    display: inline-block;
    margin-top: 4px;
">📱 Keep-Alive: ACTIVE</div>

<script>
(function() {
    // ── 1. Silent audio loop (most reliable on iOS/Android) ──
    // Generates a minimal WAV with near-zero amplitude
    function createSilentAudio() {
        const ctx = new (window.AudioContext || window.webkitAudioContext)();
        const oscillator = ctx.createOscillator();
        const gainNode = ctx.createGain();
        gainNode.gain.value = 0.001; // near-silent
        oscillator.connect(gainNode);
        gainNode.connect(ctx.destination);
        oscillator.start();
        return ctx;
    }

    let audioCtx = null;
    try { audioCtx = createSilentAudio(); } catch(e) {
        console.log("Keep-alive: AudioContext not available, using fallback only.");
    }

    // ── 2. Periodic "Connect" button click (Colab-specific fallback) ──
    const keepAliveInterval = setInterval(function() {
        // Click the connect button if it exists and says "Connect" or "Reconnect"
        const buttons = document.querySelectorAll("colab-connect-button");
        buttons.forEach(btn => {
            const shadow = btn.shadowRoot;
            if (shadow) {
                const connectBtn = shadow.querySelector("#connect");
                if (connectBtn && (connectBtn.innerText.includes("Connect") || connectBtn.innerText.includes("Reconnect"))) {
                    connectBtn.click();
                    console.log("Keep-alive: Reconnect clicked");
                }
            }
        });
    }, 60000); // every 60 seconds

    // ── 3. Resume AudioContext when tab regains focus ──
    document.addEventListener("visibilitychange", function() {
        if (!document.hidden && audioCtx && audioCtx.state === "suspended") {
            audioCtx.resume();
            console.log("Keep-alive: AudioContext resumed on tab focus");
        }
    });

    // ── 4. Ping output area to prevent idle timeout ──
    setInterval(function() {
        const el = document.getElementById("keep-alive-status");
        if (el) {
            const now = new Date().toLocaleTimeString();
            el.textContent = "📱 Keep-Alive: ACTIVE — Last ping: " + now;
        }
    }, 30000); // update status every 30s

    console.log("📱 Mobile keep-alive initialized.");
})();
</script>
'''))

print("✅ Mobile keep-alive activated!")
print("   → Silent audio loop running (prevents background tab kill)")
print("   → Auto-reconnect pinger active (60s interval)")
print("   → You can safely switch apps on your phone now.")

✅ Mobile keep-alive activated!
   → Silent audio loop running (prevents background tab kill)
   → Auto-reconnect pinger active (60s interval)
   → You can safely switch apps on your phone now.


## 🖥️ Cell 0.5 — GPU Check (Not essential)

In [ ]:
import subprocess, sys, shutil

print("=" * 50)
print("  🔍 Diagnostics")
print("=" * 50)
print(f"  Python : {sys.version.split()[0]}")

try:
    r = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
         "--format=csv,noheader,nounits"],
        capture_output=True, text=True, check=True
    )
    name, total, free = [x.strip() for x in r.stdout.strip().split(",")]
    print(f"  GPU    : {name}")
    print(f"  VRAM   : {int(total)/1024:.1f} GB total | {int(free)/1024:.1f} GB free")
    if int(total) < 8000:
        print("  ⚠️  VRAM < 8 GB — enable low_vram mode in Cell 3.")
    else:
        print("  ✅  VRAM looks good!")
except Exception as e:
    print(f"  ❌ GPU not found: {e}")
    print("  → Go to: Runtime → Change runtime type → GPU")
    sys.exit(1)

_, _, free_d = shutil.disk_usage("/")
print(f"  Disk   : {free_d // 2**30} GB free")
print("=" * 50)

  🔍 Diagnostics
  Python : 3.12.13
  GPU    : Tesla T4
  VRAM   : 15.0 GB total | 14.6 GB free
  ✅  VRAM looks good!
  Disk   : 65 GB free


## 📦 Cell 1 — Install InvokeAI
> A single `pip install` — no `git clone` of 5 GB repos.

In [ ]:
import subprocess, sys

print("🔧 Removing TensorFlow (not needed, causes conflicts)...")
subprocess.run([
    sys.executable, "-m", "pip", "uninstall", "-y", "-q",
    "tensorflow", "tensorflow-cpu", "tensorflow-gpu",
    "tf-keras", "keras"
], capture_output=True)

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "protobuf>=4.25.0"
], check=True)

print("⬇️  Installing InvokeAI (may take 2–4 minutes)...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "InvokeAI[xformers]",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121"
], check=True)

print("⬇️  Installing cloudflared for public link...")
subprocess.run([
    "wget", "-q",
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    "-O", "/usr/local/bin/cloudflared"
], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

print("\n✅ Installation complete!")

🔧 Removing TensorFlow (not needed, causes conflicts)...
⬇️  Installing InvokeAI (may take 2–4 minutes)...
⬇️  Installing cloudflared for public link...

✅ Installation complete!


## ⚙️ Cell 3 — Configure InvokeAI

In [ ]:
import os, yaml

INVOKE_ROOT = os.path.expanduser("~/invokeai")
os.makedirs(INVOKE_ROOT, exist_ok=True)

LOW_VRAM_MODE = False   # True if you have < 8 GB VRAM

config = {
    "InvokeAI": {
        "Paths": {
            "root": INVOKE_ROOT,
            "outdir": os.path.join(INVOKE_ROOT, "outputs"),
        },
        "Memory/Performance": {
            "ram": 12.0,
            "vram": 0.5 if LOW_VRAM_MODE else 2.75,
            "lazy_offload": True,
            "sequential_guidance": LOW_VRAM_MODE,
            "attention_type": "xformers",
            "attention_slice_size": "auto",
        },
        "Device": {
            "device": "cuda",
            "precision": "auto",
        },
        "Web Server": {
            "host": "0.0.0.0",
            "port": 9090,
            "allow_origins": ["*"],
        },
    }
}

cfg_path = os.path.join(INVOKE_ROOT, "invokeai.yaml")
with open(cfg_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

for d in ["outputs", "models/checkpoints/main", "models/loras",
          "models/controlnet", "models/embeddings"]:
    os.makedirs(os.path.join(INVOKE_ROOT, d), exist_ok=True)

print("✅ invokeai.yaml configured!")
print(f"   Root     : {INVOKE_ROOT}")
print(f"   Port     : 9090")
print(f"   Low VRAM : {'Enabled' if LOW_VRAM_MODE else 'Disabled'}")

✅ invokeai.yaml configured!
   Root     : /root/invokeai
   Port     : 9090
   Low VRAM : Disabled


## 🚀 Cell 4 — Start InvokeAI
> The public link will appear seconds after the server is up.

In [ ]:
import subprocess, threading, time, os, sys, re
import urllib.request, urllib.error

INVOKE_ROOT = os.path.expanduser("~/invokeai")
PORT = 9090

os.environ["INVOKEAI_ROOT"]    = INVOKE_ROOT
os.environ["PYTHONUNBUFFERED"] = "1"

print("🚀 Starting InvokeAI...")
server = subprocess.Popen(
    ["invokeai-web"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1,
    env=os.environ
)

def watch_server():
    for line in server.stdout:
        print(line, end="")

t = threading.Thread(target=watch_server, daemon=True)
t.start()

print("   ⏳ Waiting for server to start...", end="")
ready = False
for _ in range(150):
    if server.poll() is not None and server.returncode != 0:
        print(f"\n\n❌ InvokeAI crashed! Exit code: {server.returncode}")
        print("Tip: Check if the model was downloaded correctly in Cell 2.")
        raise SystemExit(1)
    try:
        r = urllib.request.urlopen(f"http://localhost:0{PORT}/api/v1/app/version", timeout=2)
        if r.status == 200:
            ready = True
            break
    except (urllib.error.URLError, ConnectionError, OSError):
        pass
    time.sleep(2)
    print(".", end="", flush=True)
print()

if not ready:
    print("\n⚠️  5-minute timeout — server did not respond.")
    raise SystemExit(1)

print("\n🌐 Creating public link (cloudflared)...")
tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

url_found = False
for line in tunnel.stdout:
    if "trycloudflare.com" in line:
        match = re.search(r"https://[\w\-]+\.trycloudflare\.com", line)
        if match:
            url = match.group(0)
            print("\n" + "=" * 52)
            print(f"  🎉  InvokeAI READY!")
            print(f"  🔗  {url}")
            print("=" * 52)
            print("  → Open the link above in your browser")
            print("=" * 52 + "\n")
            url_found = True
            break

try:
    server.wait()
except KeyboardInterrupt:
    server.terminate()
    tunnel.terminate()
    print("\n🛑 Shut down.")

🚀 Starting InvokeAI...
   ⏳ Waiting for server to start............../usr/local/lib/python3.12/dist-packages/jaxlib/plugin_support.py:71: RuntimeWarning: JAX plugin jax_cuda12_plugin version 0.7.2 is installed, but it is not compatible with the installed jaxlib version 0.7.1, so it will not be used.
  warnings.warn(
..Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
.[2026-06-10 02:57:50,846]::[InvokeAI]::INFO --> Using torch device: Tesla T4
INFO:InvokeAI:Using torch device: Tesla T4
[2026-06-10 02:57:50,870]::[InvokeAI]::INFO --> cuDNN version: 90501
INFO:InvokeAI:cuDNN version: 90501
.>> patchmatch.patch_match: INFO - Compiling and loading c extensions from "/usr/local/lib/python3.12/dist-packages/patchmatch".
INFO:patchmatch.patch_match:

---
## 🔧 Extra Utilities (Non essentials)

### 💾 Save outputs to Google Drive (optional)

In [ ]:
from google.colab import drive
import os, shutil

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/InvokeAI_Outputs/LunarPeachMix"
LOCAL_DIR = os.path.expanduser("~/invokeai/outputs")
os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.islink(LOCAL_DIR):
    if os.path.exists(LOCAL_DIR):
        shutil.copytree(LOCAL_DIR, DRIVE_DIR, dirs_exist_ok=True)
        shutil.rmtree(LOCAL_DIR)
    os.symlink(DRIVE_DIR, LOCAL_DIR)
    print(f"✅ Outputs → Drive: {DRIVE_DIR}")
else:
    print("✅ Already configured.")